# NVIDIA stock trend and frequency spectra

Run `python download_data.py` in the project directory first. The controls below select the dates, period range, and FFT smoothing for four stacked plots, including the inverse FFT reconstruction.

In [23]:
from pathlib import Path
import pandas as pd
import plotly.express as px

data_file = Path.cwd() / 'data' / 'nvda_10y.csv'
if not data_file.exists():
    data_file = Path.cwd().parent / 'data' / 'nvda_10y.csv'
if not data_file.exists():
    raise FileNotFoundError('Run python download_data.py from the project directory first.')

prices = pd.read_csv(data_file, parse_dates=['Date']).sort_values('Date')
prices[['Date', 'Close']].tail()

,Date,Close
2508,2026-09-18,222.270004
2509,2026-09-21,227.380005
2510,2026-09-22,228.869995
2511,2026-09-23,225.509995
2512,2026-09-24,222.264999


## Select a time range and spectrum settings

Changing a control redraws the adjusted price, Welch power spectrum, raw versus smoothed FFT, and inverse FFT reconstruction below. Dates are inclusive; select at least 64 trading days. The period slider changes only the visible spectrum range. The smoothing slider sets the Gaussian width in FFT frequency bins. For the inverse FFT, smoothed magnitudes are paired with the original phase, so the lower plot shows reconstructed daily log returns rather than prices. Peaks describe the selected sample and do not predict prices.

In [24]:
import ipywidgets as widgets
from IPython.display import display
from datetime import datetime

first_date = datetime.strptime("2026-01-01", "%Y-%m-%d").date()  #prices['Date'].min().date()
last_date = prices['Date'].max().date()
start_picker = widgets.DatePicker(description='From', value=first_date)
end_picker = widgets.DatePicker(description='Through', value=last_date)
period_picker = widgets.IntRangeSlider(value=(5, 128), min=2, max=512, step=1, description='Period (days)', continuous_update=False, layout=widgets.Layout(width='500px'))
smooth_picker = widgets.IntSlider(value=3, min=1, max=12, step=1, description='Smooth (bins)', continuous_update=False, layout=widgets.Layout(width='500px'))
display(widgets.VBox([widgets.HBox([start_picker, end_picker]), period_picker, smooth_picker]))

In [25]:
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.signal import welch
from plotly.subplots import make_subplots

def plot_selected_range(start_date, end_date, period_range, smooth_bins):
    if start_date is None or end_date is None:
        print('Choose both dates.')
        return
    if start_date > end_date:
        print('The start date must be on or before the end date.')
        return

    selected = prices.loc[prices['Date'].between(pd.Timestamp(start_date), pd.Timestamp(end_date))]
    if len(selected) < 64:
        print('Select at least 64 trading days within the downloaded data.')
        return

    log_return_series = np.log(selected['Close'].astype(float)).diff().dropna()
    log_returns = log_return_series.to_numpy()
    centered_returns = log_returns - log_returns.mean()
    sample_count = centered_returns.size
    min_period, max_period = period_range

    fft_frequency = np.fft.rfftfreq(sample_count, d=1.0)[1:]
    fft_coefficients = np.fft.rfft(centered_returns)
    raw_magnitude = np.abs(fft_coefficients)[1:] / sample_count
    smooth_magnitude = gaussian_filter1d(raw_magnitude, sigma=smooth_bins, mode='nearest')
    fft_spectrum = pd.DataFrame({
        'Period (trading days)': 1 / fft_frequency,
        'Frequency (cycles/trading day)': fft_frequency,
        'Raw FFT magnitude': raw_magnitude,
        'Smoothed FFT magnitude': smooth_magnitude,
    })
    fft_spectrum = fft_spectrum.sort_values('Period (trading days)')

    # Magnitude alone cannot be inverted; retain the original FFT phase.
    smoothed_coefficients = fft_coefficients.copy()
    smoothed_coefficients[1:] = (
        smooth_magnitude * sample_count * np.exp(1j * np.angle(fft_coefficients[1:]))
    )
    if sample_count % 2 == 0:
        smoothed_coefficients[-1] = smoothed_coefficients[-1].real
    reconstructed_returns = np.fft.irfft(smoothed_coefficients, n=sample_count) + log_returns.mean()
    return_series = pd.DataFrame({
        'Date': selected.loc[log_return_series.index, 'Date'].to_numpy(),
        'Original log return': log_returns,
        'IFFT after smoothing': reconstructed_returns,
    })

    segment_size = min(256, max(32, sample_count // 2))
    welch_frequency, power = welch(
        centered_returns, fs=1.0, window="hann",
        nperseg=segment_size, noverlap=segment_size // 2,
        detrend='constant', scaling='density',
    )
    welch_spectrum = pd.DataFrame({
        'Period (trading days)': 1 / welch_frequency[1:],
        'Frequency (cycles/trading day)': welch_frequency[1:],
        'Power spectral density': power[1:],
    })
    welch_spectrum = welch_spectrum.sort_values('Period (trading days)')

    price_line = px.line(selected, x='Date', y='Close', render_mode='svg').data[0]
    welch_line = px.line(
        welch_spectrum, x='Period (trading days)', y='Power spectral density',
        hover_data=['Frequency (cycles/trading day)'], render_mode='svg',
    ).data[0]
    raw_line = px.line(
        fft_spectrum, x='Period (trading days)', y='Raw FFT magnitude',
        hover_data=['Frequency (cycles/trading day)'], render_mode='svg',
    ).data[0]
    raw_line.update(name='Raw FFT', line_color='rgba(100, 110, 120, 0.45)', showlegend=True)
    smooth_line = px.line(
        fft_spectrum, x='Period (trading days)', y='Smoothed FFT magnitude',
        hover_data=['Frequency (cycles/trading day)'], render_mode='svg',
    ).data[0]
    smooth_line.update(name='Smoothed FFT', line_color='#1f77b4', showlegend=True)

    returns_line = px.line(
        return_series, x='Date', y='Original log return', render_mode='svg',
    ).data[0]
    returns_line.update(name='Original returns', line_color='rgba(100, 110, 120, 0.45)', showlegend=True)
    ifft_line = px.line(
        return_series, x='Date', y='IFFT after smoothing', render_mode='svg',
    ).data[0]
    ifft_line.update(name='IFFT after smoothing', line_color='#2ca02c', showlegend=True)

    figure = make_subplots(
        rows=4, cols=1,
        subplot_titles=('Adjusted closing price', f'Welch power spectrum ({segment_size}-day segments)', 'Raw and smoothed FFT magnitude', 'IFFT reconstruction of daily log returns'),
        row_heights=[0.28, 0.22, 0.22, 0.28], vertical_spacing=0.08,
    )
    figure.add_trace(price_line, row=1, col=1)
    figure.add_trace(welch_line, row=2, col=1)
    figure.add_trace(raw_line, row=3, col=1)
    figure.add_trace(smooth_line, row=3, col=1)
    figure.add_trace(returns_line, row=4, col=1)
    figure.add_trace(ifft_line, row=4, col=1)
    figure.update_xaxes(title_text='Date', row=1, col=1)
    figure.update_yaxes(title_text='Price (USD)', row=1, col=1)
    for row in (2, 3):
        figure.update_xaxes(title_text='Period (trading days)', range=[min_period, max_period], row=row, col=1)
    figure.update_yaxes(title_text='Welch PSD', row=2, col=1)
    figure.update_yaxes(title_text='FFT magnitude', row=3, col=1)
    figure.update_xaxes(title_text='Date', row=4, col=1)
    figure.update_yaxes(title_text='Daily log return', row=4, col=1)
    figure.update_layout(
        title=f'NVDA: {selected.Date.iloc[0]:%Y-%m-%d} to {selected.Date.iloc[-1]:%Y-%m-%d}',
        height=1400, legend=dict(orientation="h", y=-0.06),
    )
    figure.show(renderer='plotly_mimetype')

chart_output = widgets.interactive_output(
    plot_selected_range,
    {'start_date': start_picker, 'end_date': end_picker, 'period_range': period_picker, 'smooth_bins': smooth_picker},
)
display(chart_output)

Output()